# Domestic Flight Price Prediction — 03 · Models

Report sections 6–11: validation strategy, **basic models** (Linear Regression, Polynomial Regression, Ridge, Lasso, Decision Tree, per-class Ridge), **advanced models** (Random Forest, Extra Trees, LightGBM, XGBoost, CatBoost, MLP, TabM, FT-Transformer, TabPFN v2), hyperparameter tuning, the **final ensemble** and the training-set-size experiment.

All models use the features built in 02_Preprocessing (`flight_pipeline.py`) and the same 5-fold cross-validation, so every score is directly comparable. Metric: RMSE on the raw price.

In [ ]:
# What to retrain in this run. results/predictions holds the out-of-fold and test
# predictions of every model, produced by exactly this code. CPU models (scikit-learn,
# LightGBM, XGBoost, CatBoost) reproduce them bit for bit; the GPU models (neural
# networks, TabPFN) are slow and not bit-for-bit deterministic, so they are loaded
# by default. Times: Ryzen 7 7435HS laptop with an RTX 4060 (8 GB).
RETRAIN = {
    "bagged_trees": True,    # ~4 min   Random Forest, Extra Trees
    "lightgbm": True,        # ~20 min  5 variants x 3 seeds
    "xgboost": True,         # ~5 min   2 variants x 3 seeds
    "catboost": True,        # ~45 min  3 variants
    "neural_nets": False,    # ~40 min on the GPU: MLP, TabM x2, FT-Transformer x2
    "tabpfn": False,         # ~2.5 h on the GPU: 4 TabPFN v2 runs
}
RUN_OPTUNA = False           # ~1.5 h: re-run the hyperparameter searches (Section 4)

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
import sklearn, lightgbm, xgboost, catboost

import flight_pipeline as fp

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 90)
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#bdbbb3"
INK2, MUTED, GRID, BASE = "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": BASE, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2, "grid.color": GRID, "grid.linewidth": 0.6,
    "legend.frameon": False, "lines.linewidth": 2, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left"})
num0 = FuncFormatter(lambda v, _: f"{v:,.0f}")

print("python", platform.python_version(), "| scikit-learn", sklearn.__version__, "| lightgbm", lightgbm.__version__,
      "| xgboost", xgboost.__version__, "| catboost", catboost.__version__,
      "| torch", (fp.torch.__version__ + (" (CUDA)" if fp.torch.cuda.is_available() else " (CPU)")) if fp.HAVE_TORCH else "not installed")
y = fp.build_base()[2]

## 1. Validation strategy

- **5-fold cross-validation** (`KFold`, shuffle, seed 42) on all 15,589 training rows, with the **same folds for every model**, so out-of-fold (OOF) predictions can be compared and blended row by row.
- **No per-fold early stopping:** the number of boosting rounds (or epochs) is chosen once, from the validation curve averaged over the folds.
- **Test predictions come from a refit on all training rows** (Section 6 shows why the extra data matters).
- The team notebooks used a single 80/20 holdout; Section 2.7 shows why 5-fold CV is more reliable here.

In [ ]:
for k, (tr, va) in enumerate(fp.outer_folds()):
    print(f"fold {k}: {len(tr):,} training rows, {len(va):,} validation rows")

## 2. Basic models

Two feature sets (built in 02_Preprocessing): plain **one-hot** encoding of every categorical including the 1,070-level flight number (the `new.ipynb` setup), and the **engineered** features of the tuned notebooks. Every transformer sits inside a `Pipeline`, so it is re-fitted on each training fold.

In [ ]:
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor

Xb, yb = fp.basic_data()
basic_rows, sweeps, basic_oof = [], [], {}

def record(model, features, setting, oof):
    s = fp.score(oof)
    basic_rows.append({"model": model, "features": features, "setting": setting,
                       "CV RMSE": s["rmse"], "R2": s["r2"], "MAE": s["mae"], "Economy": s["rmse_economy"],
                       "Dom. Biz": s["rmse_biz_domestic"], "For. Biz": s["rmse_biz_foreign"]})
    print(f"{model:22s} [{features:10s}] {setting:22s} {fp.fmt(s)}")

### 2.1 Linear Regression and 2.2 Polynomial Regression
Polynomial regression adds degree-2 terms of `duration` and `days_left`.

In [ ]:
for fs, (cats, nums, mf) in fp.FEATURE_SETS.items():
    oof = fp.cv_predict(lambda: fp.onehot_pipe(LinearRegression(), cats, nums, min_freq=mf), Xb, yb)
    record("Linear Regression", fs, "-", oof)
    basic_oof[f"linear_{fs}"] = oof
for fs, (cats, nums, mf) in fp.FEATURE_SETS.items():
    oof = fp.cv_predict(lambda: fp.onehot_pipe(LinearRegression(), cats, nums, poly=True, min_freq=mf), Xb, yb)
    record("Polynomial Regression", fs, "degree 2 on numerics", oof)

### 2.3 Ridge (L2) and 2.4 Lasso (L1)
The penalty strength α is chosen by CV RMSE.

In [ ]:
for kind, Est, grid in (("Ridge", Ridge, [0.1, 0.3, 1, 3, 10, 30, 100]),
                        ("Lasso", Lasso, [1, 3, 10, 30, 100, 300, 1000])):
    for fs, (cats, nums, mf) in fp.FEATURE_SETS.items():
        best = None
        for a in grid:
            kw = {"alpha": a} | ({"max_iter": 20000, "tol": 1e-3} if kind == "Lasso" else {})
            oof = fp.cv_predict(lambda: fp.onehot_pipe(Est(**kw), cats, nums, min_freq=mf), Xb, yb)
            r = fp.rmse(yb, oof)
            sweeps.append({"model": kind, "features": fs, "alpha": a, "CV RMSE": r})
            if best is None or r < best[1]:
                best = (a, r, oof)
        record(kind, fs, f"alpha={best[0]:g}", best[2])
        basic_oof[f"{kind.lower()}_{fs}"] = best[2]

### 2.5 Decision Tree
Ordinal-encoded features; maximum depth × minimum leaf size chosen by CV RMSE.

In [ ]:
tree_sweep = []
for fs, (cats, nums, _) in fp.FEATURE_SETS.items():
    best = None
    for depth in (4, 6, 8, 10, 12, 15, 20, None):
        for leaf in (1, 5, 10, 20, 40):
            oof = fp.cv_predict(lambda: fp.tree_pipe(DecisionTreeRegressor(max_depth=depth, min_samples_leaf=leaf,
                                                                             random_state=0), cats, nums), Xb, yb)
            r = fp.rmse(yb, oof)
            tree_sweep.append({"features": fs, "max_depth": depth or "none", "min_leaf": leaf, "CV RMSE": r})
            if best is None or r < best[1]:
                best = ((depth, leaf), r, oof)
    record("Decision Tree", fs, f"depth {best[0][0]}, min leaf {best[0][1]}", best[2])
    basic_oof[f"tree_{fs}"] = best[2]

In [ ]:
sw = pd.DataFrame(sweeps)
ts = pd.DataFrame(tree_sweep)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for fs, col in (("onehot", BLUE), ("engineered", ORANGE)):
    s = sw[(sw["model"] == "Ridge") & (sw["features"] == fs)]
    axes[0].plot(s["alpha"], s["CV RMSE"], color=col, marker="o", markersize=5, markeredgecolor="white", label=fs)
axes[0].set_xscale("log")
axes[0].set_title("Ridge: penalty strength")
axes[0].set_xlabel("alpha (log scale)")
axes[0].set_ylabel("CV RMSE")
axes[0].legend()
s = ts[(ts["features"] == "onehot") & (ts["min_leaf"] == 5)]
xs = [d if d != "none" else 26 for d in s["max_depth"]]
axes[1].plot(xs, s["CV RMSE"], color=BLUE, marker="o", markersize=5, markeredgecolor="white")
axes[1].set_xticks([4, 6, 8, 10, 12, 15, 20, 26], ["4", "6", "8", "10", "12", "15", "20", "none"])
axes[1].set_title("Decision tree: depth (min leaf 5, one-hot features)")
axes[1].set_xlabel("max_depth")
for ax in axes:
    ax.yaxis.set_major_formatter(num0)
    ax.grid(axis="y")
plt.show()

### 2.6 Per-class Ridge (the linear project's final model)
Separate Ridge models for Economy and Business on the L5 feature ladder (one-hot, engineered numerics, class interactions, splines, target-encoded flight).

In [ ]:
oof_pc, test_pc = fp.run_perclass_ridge()          # also saved for the ensemble (results/predictions)
record("Per-class Ridge", "L5 ladder", "alpha=0.1, one per cabin", oof_pc)

### Summary of basic models

In [ ]:
basic = pd.DataFrame(basic_rows)
basic.to_csv(fp.RESULTS / "basic_models.csv", index=False)
display(basic.style.format({c: "{:,.0f}" for c in ["CV RMSE", "MAE", "Economy", "Dom. Biz", "For. Biz"]} | {"R2": "{:.4f}"})
        .hide(axis="index"))

order = ["Linear Regression", "Polynomial Regression", "Ridge", "Lasso", "Decision Tree"]
fig, ax = plt.subplots(figsize=(9, 3.4))
yy = np.arange(len(order))
for i, (fs, col) in enumerate((("onehot", BLUE), ("engineered", ORANGE))):
    v = [basic[(basic["model"] == m) & (basic["features"] == fs)]["CV RMSE"].iloc[0] for m in order]
    ax.barh(yy + (i - 0.5) * 0.38, v, height=0.34, color=col, label=f"{fs} features")
pc = basic.loc[basic["model"] == "Per-class Ridge", "CV RMSE"].iloc[0]
ax.axvline(pc, color=INK2, linewidth=1)
ax.text(pc - 50, -0.95, f"per-class Ridge {pc:,.0f}", ha="right", color=INK2, fontsize=8)
ax.set_yticks(yy, order)
ax.invert_yaxis()
ax.set_xlim(10000, 13800)
ax.xaxis.set_major_formatter(num0)
ax.set_xlabel("5-fold CV RMSE (axis starts at 10,000)")
ax.grid(axis="x")
ax.set_axisbelow(True)
ax.legend(loc="lower right")
plt.show()

- Engineered features matter more than the choice of linear model: they cut Linear Regression's RMSE by about 1,400, while OLS, Ridge and Lasso differ by under 200.
- Ridge is the best plain linear model; Lasso is slightly worse because L1 zeroes correlated dummy columns one by one instead of shrinking them together.
- The decision tree beats every linear model on raw features and is the best basic model for Economy, but weak on foreign Business. Shallow trees under-fit and unlimited depth over-fits.
- Fitting the two cabins separately (per-class Ridge) is the best linear approach.

### 2.7 Why 5-fold CV and not one holdout

In [ ]:
from sklearn.model_selection import train_test_split
_, hold = train_test_split(np.arange(len(yb)), test_size=0.2, random_state=42)   # the notebooks' holdout split
o = basic_oof["linear_engineered"]
print(f"engineered Linear Regression, scored on all rows (5-fold CV):      {fp.rmse(yb, o):8,.0f}")
print(f"the same predictions, scored only on the notebooks' holdout rows:  {fp.rmse(yb[hold], o[hold]):8,.0f}   (notebook: 11,025)")
print(f"fares above 200,000: {100 * (yb > 200000).mean():.2f}% of all rows, {100 * (yb[hold] > 200000).mean():.2f}% of the holdout rows")

The holdout is simply an easier sample (fewer extreme fares), so its scores look better than the 5-fold CV ones.

### 2.8 Team notebook experiments (80/20 holdout)

| Notebook model (holdout) | RMSE | MAE | R² |
|---|---|---|---|
| Linear Regression, one-hot incl. flight (`new.ipynb`) | 12,137 | 5,759 | 0.8955 |
| Ridge α = 1, one-hot incl. flight | 12,125 | 5,735 | 0.8957 |
| Polynomial (degree 2) + one-hot | 12,069 | 5,804 | 0.8967 |
| Lasso α = 100, one-hot | 14,251 | 7,202 | 0.8559 |
| Linear Regression, engineered | 11,025 | 5,111 | 0.9137 |
| Ridge α = 1, engineered | 10,986 | 5,081 | 0.9144 |
| Extra Trees (tuned: 900 trees, min leaf 2) | 10,166 | 3,125 | 0.9267 |
| CatBoost (depth 8) | 10,551 | 4,044 | 0.9210 |
| XGBoost (depth 7, 900 trees) | 9,977 | 3,629 | 0.9294 |
| XGBoost tuned (depth 6, 1,100 trees) | 9,872 | 3,688 | 0.9309 |
| Blend 65% XGBoost + 35% Extra Trees | 9,751 | 3,375 | 0.9325 |

They are lower than the CV numbers above because of the easier split, but they rank the models broadly the same way.

## 3. Advanced models

All advanced models use the full feature set of 02_Preprocessing (142 columns per row, target statistics nested inside the folds). Each run saves its OOF and test predictions to `results/predictions/`.

In [ ]:
results = {}

def get(name, retrain, train):
    # train (and save) a model, or load its saved predictions
    before = np.load(fp.PRED_DIR / f"{name}.npz")["oof"] if fp.is_saved(name) else None
    if retrain or before is None:
        train()
        if before is not None:
            after = np.load(fp.PRED_DIR / f"{name}.npz")["oof"]
            print(f"{'':28s} -> reproduces the saved run: max |difference| = {np.abs(after - before).max():.2g}")
    p = fp.load_preds([name])[name]
    if not (retrain or before is None):
        print(f"{name:28s} loaded saved predictions   {fp.fmt(fp.score(p['oof']))}")
    results[name] = p
    return p

get("linear_ridge_perclass", False, None)      # trained in Section 2.6

### 3.1 Bagged trees: Random Forest and Extra Trees
Averaging many deep trees grown on bootstrap samples (Random Forest) or with random split thresholds (Extra Trees).

In [ ]:
get("rf_msl3_mf033", RETRAIN["bagged_trees"],
    lambda: fp.run_bagged("rf_msl3_mf033", "rf", dict(n_estimators=500, min_samples_leaf=3, max_features=0.33)))
get("et_msl3_mf05", RETRAIN["bagged_trees"],
    lambda: fp.run_bagged("et_msl3_mf05", "et", dict(n_estimators=500, min_samples_leaf=3, max_features=0.5)));

### 3.2 LightGBM
Three distinct top trials of the Optuna search (Section 4) with 3 seeds each, plus two variants for diversity: all categoricals including the flight and itinerary columns, and a Tweedie objective.

In [ ]:
T9 = {"objective": "regression", "learning_rate": 0.04414516481239043, "num_leaves": 196, "min_child_samples": 4,
      "reg_lambda": 0.0053024228766843555, "reg_alpha": 0.09074256288983863, "path_smooth": 42.80967380043114,
      "colsample_bytree": 0.4694386900580503, "subsample": 0.8360677737029393, "min_split_gain": 1238.6737172447354,
      "cat_smooth": 2.98727419956384, "min_data_per_group": 43, "extra_trees": True}
T13 = {"objective": "regression", "learning_rate": 0.031518317518968955, "num_leaves": 196, "min_child_samples": 18,
       "reg_lambda": 0.0015099953308448232, "reg_alpha": 0.013247982591501046, "path_smooth": 0.009988882618951256,
       "colsample_bytree": 0.4715679071852956, "subsample": 0.711535815890973, "min_split_gain": 366.04771189292666,
       "cat_smooth": 1.715777229558653, "min_data_per_group": 48, "extra_trees": True}
T12 = {"objective": "regression", "learning_rate": 0.04264630415360833, "num_leaves": 105, "min_child_samples": 24,
       "reg_lambda": 3.0809133283035437, "reg_alpha": 0.4878431250899203, "path_smooth": 0.21333820765793288,
       "colsample_bytree": 0.4871893153028636, "subsample": 0.8830871455387921, "min_split_gain": 163.85111197111286,
       "cat_smooth": 25.837213560978167, "min_data_per_group": 49, "extra_trees": False}
LOW, HIGH, SEEDS = fp.LOW_CATS, fp.HIGH_CATS, (0, 1, 2)
lgb_models = [  # (name, parameters, categoricals, round cap)
    ("lgb_tuned_t9", T9, LOW, 1000),
    ("lgb_tuned_t13", T13, LOW, 1000),
    ("lgb_tuned_t12", T12, LOW, 1000),
    ("lgb_tuned_allcats", T13, LOW + HIGH, 1000),
    ("lgb_tuned_tweedie", {**T13, "objective": "tweedie", "tweedie_variance_power": 1.5}, LOW, 4000),
]
for name, params, cats, cap in lgb_models:
    get(name, RETRAIN["lightgbm"], lambda: fp.run_gbdt(name, "lgb", params, cats=cats, n_rounds=cap, seeds=SEEDS))

### 3.3 XGBoost
The Optuna winner (shallow depth-4 trees with a Tweedie objective) and a hand-set deep variant.

In [ ]:
XGB_TUNED = {"objective": "reg:tweedie", "eta": 0.03576188103831809, "max_depth": 4, "min_child_weight": 10.377151651186676,
             "reg_lambda": 0.027366936822699205, "reg_alpha": 0.06913034910547554, "gamma": 0.4010477329149743,
             "colsample_bytree": 0.5119275726144146, "subsample": 0.7816222479586502, "max_bin": 256}
XGB_D10 = dict(eta=0.04, max_depth=10, min_child_weight=3, reg_lambda=1.0, gamma=300.0, colsample_bytree=0.5, subsample=0.75)
for name, params in (("xgb_tuned", XGB_TUNED), ("xgb_d10", XGB_D10)):
    get(name, RETRAIN["xgboost"], lambda: fp.run_gbdt(name, "xgb", params, cats=LOW, n_rounds=1000, seeds=SEEDS))

### 3.4 CatBoost
Every categorical, including the 1,000+-level flight and itinerary columns, through CatBoost's ordered target statistics.

In [ ]:
cat_models = [
    ("cat_allcats", dict(learning_rate=0.08), 2000),
    ("cat_d8", dict(learning_rate=0.06, depth=8, l2_leaf_reg=5.0), 2500),
    ("cat_d6_rs2", dict(learning_rate=0.06, depth=6, random_strength=2.0, bagging_temperature=0.5, one_hot_max_size=8), 2500),
]
for name, params, cap in cat_models:
    get(name, RETRAIN["catboost"], lambda: fp.run_gbdt(name, "cat", params, cats=LOW + HIGH, n_rounds=cap, seeds=(0,)))

### 3.5 Neural networks (PyTorch, GPU)
- **MLP** with learned embeddings for the low-cardinality categoricals (embeddings for the 1,000+-level columns over-fitted).
- **TabM** (ICLR 2025): an efficient ensemble of 32 MLPs trained together.
- **FT-Transformer:** every feature becomes a token, self-attention mixes them. Two configurations; attention over-fits on 12k rows.

In [ ]:
curated_drop = [c for c in fp.feature_lists(fp.fold_data(0)[0])[1] if c not in fp.TABPFN_NUMS]
nn_models = [
    ("mlp_lowcats", "mlp", dict(seeds=(0, 1, 2), epochs=25, lr=1e-3, wd=1e-4)),
    ("tabm_noemb_d256", "tabm", dict(seeds=(0, 1, 2), epochs=30, lr=2e-3, wd=3e-4,
                                     model_kw=dict(n_blocks=3, d_block=256, dropout=0.1, d_emb=0))),
    ("tabm_noemb_d512_n4", "tabm", dict(seeds=(0, 1), epochs=60, lr=1e-3, wd=3e-4,
                                        model_kw=dict(n_blocks=4, d_block=512, dropout=0.2, d_emb=0))),
    ("ftt_d96_lowcats", "ftt", dict(seeds=(0,), epochs=30, lr=3e-4, wd=1e-5)),
    ("ftt_d64_curated", "ftt", dict(seeds=(0, 1), epochs=40, lr=2e-4, wd=1e-4, drop=curated_drop,
                                    model_kw=dict(d=64, n_layers=2, heads=8, attn_drop=0.3, ffn_drop=0.2))),
]
for name, arch, kw in nn_models:
    get(name, RETRAIN["neural_nets"], lambda: fp.run_nn(name, arch, cats=LOW, **kw))

### 3.6 TabPFN v2
A transformer pretrained on millions of synthetic tables to approximate Bayesian prediction. One model per cabin, so each training fold stays under its 10,000-row design limit; 43 curated features to fit the 8 GB GPU.

In [ ]:
tabpfn_models = [
    ("tabpfn_v2_perclass_raw", dict(n_estimators=4, seed=0)),
    ("tabpfn_v2_perclass_raw_e8_s1", dict(n_estimators=8, seed=1)),
    ("tabpfn_v2_perclass_raw_e8_s3_xp", dict(n_estimators=8, seed=3, nums=fp.TABPFN_NUMS_XP)),   # + composite features
    ("tabpfn_v2_perseg_raw_e8_s2", dict(n_estimators=8, seed=2, split="segment")),              # foreign Business on its own
]
for name, kw in tabpfn_models:
    get(name, RETRAIN["tabpfn"], lambda: fp.run_tabpfn(name, **kw))

### 3.7 Summary of advanced models
Segment columns are the RMSE inside each segment; the foreign-Business column decides the overall score.

In [ ]:
summary = pd.DataFrame([{"model": n, "family": fp.family(n), "CV RMSE": p["meta"]["rmse"], "R2": p["meta"]["r2"],
                         "MAE": p["meta"]["mae"], "Economy": p["meta"]["rmse_economy"],
                         "Dom. Biz": p["meta"]["rmse_biz_domestic"], "For. Biz": p["meta"]["rmse_biz_foreign"]}
                        for n, p in results.items()]).sort_values("CV RMSE").reset_index(drop=True)
summary.style.format({c: "{:,.0f}" for c in ["CV RMSE", "MAE", "Economy", "Dom. Biz", "For. Biz"]} | {"R2": "{:.4f}"})

TabPFN v2 is the best single model, and best on foreign Business: it averages small, noisy groups more sensibly than trees fitted from scratch. A transformer trained from scratch (FT-Transformer) is among the worst.

## 4. Hyperparameter tuning (Optuna)

Grid searches tuned the basic models (Section 2). For LightGBM (32 trials) and XGBoost (20 trials), Optuna's TPE sampler minimised the same 5-fold CV RMSE. The searches are stored in `results/optuna.db`; set `RUN_OPTUNA = True` to repeat them.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
STORAGE = f"sqlite:///{(fp.RESULTS / 'optuna.db').as_posix()}"

def lgb_space(t):
    obj = t.suggest_categorical("objective", ["regression", "poisson", "tweedie"])
    p = {"objective": obj,
         "learning_rate": t.suggest_float("learning_rate", 0.03, 0.12, log=True),
         "num_leaves": t.suggest_int("num_leaves", 15, 255, log=True),
         "min_child_samples": t.suggest_int("min_child_samples", 2, 60, log=True),
         "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 100, log=True),
         "reg_alpha": t.suggest_float("reg_alpha", 1e-3, 10, log=True),
         "path_smooth": t.suggest_float("path_smooth", 1e-3, 50, log=True),
         "colsample_bytree": t.suggest_float("colsample_bytree", 0.3, 1.0),
         "subsample": t.suggest_float("subsample", 0.5, 1.0),
         "min_split_gain": t.suggest_float("min_split_gain", 1e-3, 1e5, log=True),
         "cat_smooth": t.suggest_float("cat_smooth", 1, 100, log=True),
         "min_data_per_group": t.suggest_int("min_data_per_group", 5, 100, log=True),
         "extra_trees": t.suggest_categorical("extra_trees", [False, True])}
    if obj == "tweedie":
        p["tweedie_variance_power"] = t.suggest_float("tweedie_variance_power", 1.05, 1.9)
    return p

def xgb_space(t):
    return {"objective": t.suggest_categorical("objective", ["reg:squarederror", "count:poisson", "reg:tweedie"]),
            "eta": t.suggest_float("eta", 0.02, 0.12, log=True),
            "max_depth": t.suggest_int("max_depth", 3, 10),
            "min_child_weight": t.suggest_float("min_child_weight", 0.5, 50, log=True),
            "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 100, log=True),
            "reg_alpha": t.suggest_float("reg_alpha", 1e-3, 10, log=True),
            "gamma": t.suggest_float("gamma", 1e-3, 1e5, log=True),
            "colsample_bytree": t.suggest_float("colsample_bytree", 0.3, 1.0),
            "subsample": t.suggest_float("subsample", 0.5, 1.0),
            "max_bin": t.suggest_categorical("max_bin", [64, 128, 256, 512])}

def objective(kind, space, cap):
    def f(trial):
        r = fp.run_gbdt(f"{kind} trial {trial.number}", kind, space(trial), cats=fp.LOW_CATS, n_rounds=cap,
                        refit=False, save=False, verbose=False)
        trial.set_user_attr("n_iter", r["n_iter"])
        return r["score"]["rmse"]
    return f

if RUN_OPTUNA:
    for kind, space, n_trials, cap in (("lgb", lgb_space, 32, 3000), ("xgb", xgb_space, 20, 2500)):
        study = optuna.create_study(study_name=f"{kind}_rerun", storage=STORAGE, direction="minimize",
                                    load_if_exists=True, sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=12))
        study.optimize(objective(kind, space, cap), n_trials=n_trials)

for label, study_name in (("LightGBM", "lgb_v2"), ("XGBoost", "xgb_v2")):
    study = optuna.load_study(study_name=study_name, storage=STORAGE)
    df = study.trials_dataframe(attrs=("number", "value", "params")).dropna(subset=["value"])
    df.columns = [c.replace("params_", "") for c in df.columns]
    print(f"{label}: {len(df)} trials, best CV RMSE {df['value'].min():,.1f}")
    display(df.sort_values("value").head(5).rename(columns={"value": "CV RMSE"}).round(4))

The LightGBM search settled on big trees (~200 leaves) with small leaves, randomised split thresholds (`extra_trees`) and ~45% of the columns per tree; trials 9, 13 and 12 are used in Section 3.2. The XGBoost winner uses shallow depth-4 trees with a Tweedie objective.

## 5. Final model: ensemble

The OOF predictions of all models line up row by row, so blends can be scored honestly. Blends with fitted weights are scored by **nested CV** (weights fitted on four folds, applied to the fifth). The **twin pull** moves a prediction toward the mean price of its exact twins in the training data (same itinerary, cabin and day); its weight is also chosen by nested CV.

In [ ]:
preds = fp.load_preds()
names = sorted(preds, key=lambda n: preds[n]["meta"]["rmse"])
print(f"{len(names)} models; the 10 best: {', '.join(names[:10])}")
P = np.column_stack([preds[n]["oof"] for n in names])
PT = np.column_stack([preds[n]["test"] for n in names])
oof_tw, test_tw = fp.twin_arrays()

blend_rows, blends = [], {}
def add(label, oof, test, note=""):
    s = fp.score(oof)
    blends[label] = (oof, test)
    blend_rows.append({"blend": label, "CV RMSE": s["rmse"], "R2": s["r2"], "MAE": s["mae"],
                       "For. Biz": s["rmse_biz_foreign"], "note": note})

def add_with_twin(label, oof, test, note=""):
    o, t, w = fp.twin_pull(oof, test, oof_tw, test_tw)
    add(label + " + twin pull", o, t, (note + "; " if note else "") + f"twin weight {w}")

top10 = (P[:, :10].mean(1), PT[:, :10].mean(1))
add_with_twin("Average of the 10 best", *top10, "equal weights")
fs_oof, fs_test, used = fp.family_strong(names, preds)
add_with_twin("Equal average of strong families", fs_oof, fs_test, "; ".join(used))
add_with_twin("Average of the 5 best", P[:, :5].mean(1), PT[:, :5].mean(1))
add_with_twin("Convex weights over all models", fp.nested_blend(P, y, fp.fit_convex), PT @ fp.fit_convex(P, y))
add_with_twin("NNLS weights", fp.nested_blend(P, y, fp.fit_nnls), PT @ fp.fit_nnls(P, y))
add("Average of the 10 best, no twin pull", *top10)
add(f"Best single model ({names[0]})", preds[names[0]]["oof"], preds[names[0]]["test"])
blend_table = pd.DataFrame(blend_rows).sort_values("CV RMSE").reset_index(drop=True)
blend_table.style.format({"CV RMSE": "{:,.1f}", "MAE": "{:,.0f}", "For. Biz": "{:,.0f}", "R2": "{:.4f}"})

In [ ]:
final_oof, final_test = blends["Average of the 10 best + twin pull"]
best_oof = preds[names[0]]["oof"]

# paired bootstrap: is the blend reliably better than the best single model?
rng = np.random.default_rng(1)
gaps = []
for _ in range(2000):
    i = rng.integers(0, len(y), len(y))
    gaps.append(fp.rmse(y[i], best_oof[i]) - fp.rmse(y[i], final_oof[i]))
gaps = np.array(gaps)
print(f"blend vs best single model: {gaps.mean():,.0f} RMSE better on average (bootstrap SD {gaps.std():,.0f}), "
      f"better in {np.mean(gaps > 0):.0%} of resamples")

# where the final model's error lives
seg = fp.segments()
err = (final_oof - y) ** 2
for label, m in (("Economy", seg["economy"]), ("Domestic Business", seg["biz_domestic"]), ("Foreign Business", seg["biz_foreign"])):
    print(f"{label:18s} {m.mean():5.1%} of rows   {err[m].sum() / err.sum():5.1%} of the squared error")

- **Equal averages beat fitted weights:** with half of the squared error in a few dozen random fare spikes, fitted weights chase the spikes and do not transfer.
- **The twin pull is worth about 60 RMSE:** TabPFN smooths across rows and does not exploit exact duplicates as sharply as a lookup does.
- **Final model = equal average of the 10 best models** (3 TabPFN, 4 LightGBM, 2 TabM, 1 XGBoost), then `0.2 × prediction + 0.8 × twin price` where a twin exists.

In [ ]:
rng = np.random.default_rng(0)
i = rng.choice(len(y), 3500, replace=False)
biz = fp.build_base()[0]["is_business"].to_numpy()[:len(y)][i] == 1
fig, ax = plt.subplots(figsize=(8, 4.2))
lim = (3000, 350000)
ax.plot(lim, lim, color=BASE, linewidth=1)
ax.scatter(y[i][~biz], final_oof[i][~biz], s=9, color=BLUE, alpha=0.55, linewidth=0, label="Economy")
ax.scatter(y[i][biz], final_oof[i][biz], s=9, color=ORANGE, alpha=0.55, linewidth=0, label="Business")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lim)
ax.set_ylim(lim)
ticks = [5000, 10000, 20000, 50000, 100000, 200000]
kfmt = FuncFormatter(lambda v, _: f"{v / 1000:,.0f}K")
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.xaxis.set_major_formatter(kfmt)
ax.yaxis.set_major_formatter(kfmt)
ax.minorticks_off()
ax.grid(True)
ax.set_xlabel("Actual price (log scale)")
ax.set_ylabel("Predicted, out-of-fold")
ax.legend(loc="upper left", markerscale=2)
ax.set_title("Final ensemble: out-of-fold predictions vs actual (3,500-row sample)")
plt.show()

In [ ]:
# best configuration of every model family, against the final ensemble
best = {"Final blend: top-10 average + twin pull": blend_table.loc[0, "CV RMSE"]}
labels = {"tabpfn": "TabPFN v2", "lgb": "LightGBM (Optuna-tuned)", "tabm": "TabM (neural ensemble)",
          "xgb": "XGBoost (tuned)", "cat": "CatBoost", "et": "Extra Trees", "mlp": "MLP (embeddings)",
          "rf": "Random Forest", "ftt": "FT-Transformer"}
for fam, label in labels.items():
    best[label] = summary.loc[summary["family"] == fam, "CV RMSE"].min()
basic_best = {"Per-class Ridge": "Per-class Ridge", "Decision Tree": "Decision Tree", "Ridge": "Ridge",
              "Lasso": "Lasso", "Linear Regression": "Linear Regression"}
for label, m in basic_best.items():
    best[label] = basic.loc[basic["model"] == m, "CV RMSE"].min()
items = sorted(best.items(), key=lambda kv: kv[1])
fig, ax = plt.subplots(figsize=(9, 4.8))
yy = np.arange(len(items))
colours = [ORANGE if n.startswith("Final") else (GREY if n in basic_best else BLUE) for n, _ in items]
ax.barh(yy, [v for _, v in items], height=0.62, color=colours)
for k, (n, v) in enumerate(items):
    if n.startswith(("Final", "TabPFN", "Linear")):
        ax.text(v + 40, k, f"{v:,.0f}", va="center", color=INK2, fontsize=8)
ax.set_yticks(yy, [n for n, _ in items])
ax.invert_yaxis()
ax.set_xlim(9000, 12300)
ax.xaxis.set_major_formatter(num0)
ax.set_xlabel("5-fold CV RMSE, best configuration per family (axis starts at 9,000)")
ax.grid(axis="x")
ax.set_axisbelow(True)
ax.legend(handles=[Patch(color=GREY, label="Basic models"), Patch(color=BLUE, label="Advanced models"),
                   Patch(color=ORANGE, label="Final ensemble")], loc="upper right")
plt.show()

## 6. Experiment: size of the training set

LightGBM retrained on 60%, 80% and 100% of each training fold (48%, 64% and 80% of all rows), scored on the same validation rows.

In [ ]:
lc = fp.learning_curve()
lc["change"] = lc["cv_rmse"].diff()
display(lc.style.format({"share_of_rows": "{:.0%}", "cv_rmse": "{:,.0f}", "change": "{:+,.0f}"}))
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(100 * lc["share_of_rows"], lc["cv_rmse"], color=BLUE, marker="o", markersize=7, markeredgecolor="white")
for s_, r_ in zip(100 * lc["share_of_rows"], lc["cv_rmse"]):
    ax.text(s_, r_ + 60, f"{r_:,.0f}", ha="center", color=INK2, fontsize=8)
ax.set_xticks(100 * lc["share_of_rows"], [f"{s_:.0%}" for s_ in lc["share_of_rows"]])
ax.yaxis.set_major_formatter(num0)
ax.set_xlabel("Share of the labelled rows used for training")
ax.set_ylabel("CV RMSE")
ax.grid(axis="y")
ax.set_title("RMSE keeps falling as data is added")
plt.show()

The curve is still steep, because more rows sharpen every itinerary's average. So every final model is **refitted on all 15,589 training rows** before predicting the test set.

## 7. Test-set predictions of the final model

In [ ]:
path = fp.write_predictions(final_test, fp.ROOT / "final_predictions.csv")
out = pd.read_csv(path)
print(f"wrote {path.name}: {out.shape[0]:,} rows, mean price {out['price'].mean():,.0f}")
out.head()

## Summary

| | CV RMSE | R² |
|---|---|---|
| Linear Regression (one-hot) | ≈ 13,272 | 0.885 |
| Best linear model (per-class Ridge) | ≈ 11,486 | 0.914 |
| Best single model (TabPFN v2) | ≈ 9,627 | 0.940 |
| **Final ensemble** (top-10 average + twin pull) | **≈ 9,474** | **0.941** |

The final ensemble is 17.5% better than the best linear model. The remaining error is close to the noise floor set by random fare spikes (01_EDA, Section 7).